# Pascal VOC 2007 Object Detection — LWCE Loss Comparison

Faster-RCNN (ResNet50+FPN, COCO pretrained) on VOC 2007.  
Comparing 6 loss functions: `ce`, `pwce`, `lwce`, `plwce`, `cb`, `focal`.

In [ ]:
# Cell 0: Environment + Setup
!pip install -q torch torchvision optuna pandas openpyxl

import os, sys, json, warnings
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import pandas as pd
import importlib.util
from collections import defaultdict

warnings.filterwarnings('ignore')

# ── GitHub repo ────────────────────────────────────────────────────
REPO_PATH = '/content/imbalanced-data-LWCE'
if not os.path.exists(REPO_PATH):
    print('Cloning...')
    !git clone https://github.com/gseungho/imbalanced-data-LWCE.git {REPO_PATH}

# ── Load classification loss functions ────────────────────────────
get_clf_loss = None
try:
    spec = importlib.util.spec_from_file_location(
        'clf', f'{REPO_PATH}/image_classification/custom_losses.py'
    )
    clf = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(clf)
    get_clf_loss = clf.get_clf_loss
    print('\u2713 Loss functions loaded!')
except Exception as e:
    print(f'Warning: {e}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
torch.cuda.empty_cache()

# ── Constants ──────────────────────────────────────────────────────
DATASET_NAME      = 'VOC2007'
NUM_CLASSES       = 21        # 20 VOC classes + 1 background (0=bg)
BATCH_SIZE        = 4
NUM_WORKERS       = 0
MAX_TRAIN_BATCHES = 400       # 400x4=1600 images/epoch (32% of trainval)
PROXY_EPOCHS      = 5
FINAL_EPOCHS      = 20
RESULTS_DIR = '/content/gdrive/MyDrive/imbalanced-data-LWCE/image_detection/results/VOC2007'
os.makedirs(RESULTS_DIR, exist_ok=True)

# VOC 20 class names; label = index+1, background = 0
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird',       'boat',        'bottle',
    'bus',       'car',     'cat',        'chair',       'cow',
    'diningtable','dog',    'horse',      'motorbike',   'person',
    'pottedplant','sheep',  'sofa',       'train',       'tvmonitor'
]
VOC_CLASS_TO_LABEL = {cls: idx + 1 for idx, cls in enumerate(VOC_CLASSES)}
# background=0, aeroplane=1, ..., tvmonitor=20

print(f'VOC classes: {len(VOC_CLASSES)} \u2192 labels 1-{len(VOC_CLASSES)}')
print(f'Results dir: {RESULTS_DIR}')

In [ ]:
# Cell 1: Load VOC 2007
from torchvision.datasets import VOCDetection
from torchvision.transforms import Normalize
import torchvision.transforms.functional as TF

class _VOCTransform:
    """PIL image -> normalized tensor; target returned as-is."""
    def __call__(self, img):
        t = TF.to_tensor(img)
        t = Normalize(mean=[0.485, 0.456, 0.406],
                      std=[0.229, 0.224, 0.225])(t)
        return t

def load_voc_data(batch_size=4, num_workers=0):
    print('Loading Pascal VOC 2007...')
    transform = _VOCTransform()

    train_dataset = VOCDetection(
        root='/content/VOC', year='2007', image_set='trainval',
        download=True, transform=transform
    )
    test_dataset = VOCDetection(
        root='/content/VOC', year='2007', image_set='test',
        download=True, transform=transform
    )

    # class_counts[i] = # training images containing class i (0-indexed, 20 classes)
    print('Computing class distribution...')
    class_counts = [0] * 20
    for _, target in train_dataset:
        objects = target['annotation'].get('object', [])
        if isinstance(objects, dict):
            objects = [objects]
        seen = set()
        for obj in objects:
            label = VOC_CLASS_TO_LABEL.get(obj['name'], 0)
            if label > 0:
                seen.add(label - 1)  # 0-indexed
        for cidx in seen:
            class_counts[cidx] += 1

    print(f'\u2713 VOC 2007: {len(train_dataset)} trainval, {len(test_dataset)} test')
    ir = max(class_counts) / max(min(class_counts), 1)
    print(f'Class distribution: min={min(class_counts)}, max={max(class_counts)}, '
          f'mean={np.mean(class_counts):.0f}, imbalance={ir:.1f}:1')

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              num_workers=num_workers, shuffle=True,
                              collate_fn=lambda x: x)
    val_loader   = DataLoader(test_dataset,  batch_size=batch_size,
                              num_workers=num_workers, shuffle=False,
                              collate_fn=lambda x: x)
    return train_loader, val_loader, class_counts

torch.cuda.empty_cache()
train_loader, val_loader, class_counts = load_voc_data(BATCH_SIZE, NUM_WORKERS)

In [ ]:
# Cell 2: Class Distribution Visualization
def visualize_distribution(class_counts):
    counts = np.array(class_counts)
    order  = np.argsort(counts)[::-1]
    q1, q3 = np.percentile(counts, [33, 66])
    colors = ['#2ecc71' if c >= q3 else '#f39c12' if c >= q1 else '#e74c3c'
              for c in counts[order]]

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(range(20), counts[order], color=colors, alpha=0.85)
    ax.set_xticks(range(20))
    ax.set_xticklabels([VOC_CLASSES[i] for i in order],
                       rotation=45, ha='right', fontsize=9)
    ax.set_yscale('log')
    ax.set_ylabel('# Training Images (log scale)')
    ax.set_title('VOC 2007 Class Distribution  \u2014  Green: Head / Orange: Mid / Red: Tail')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/class_distribution.png', dpi=100, bbox_inches='tight')
    plt.show()

    head = sum(1 for c in counts if c >= q3)
    mid  = sum(1 for c in counts if q1 <= c < q3)
    tail = sum(1 for c in counts if c < q1)
    print(f'Head (\u2265{q3:.0f} imgs): {head} classes')
    print(f'Mid  ({q1:.0f}\u2013{q3:.0f} imgs): {mid} classes')
    print(f'Tail (<{q1:.0f} imgs): {tail} classes')
    print(f'Imbalance ratio: {max(counts)/max(min(counts),1):.1f}:1')

visualize_distribution(class_counts)

In [ ]:
# Cell 3: Model & Evaluation Functions

def build_detector(num_classes):
    """COCO pretrained Faster-RCNN; replace RoI head for num_classes."""
    from torchvision.models.detection import (
        fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
    )
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


def convert_voc_target(voc_target, device=DEVICE):
    """
    Parse VOC XML annotation dict into Faster-RCNN target format.
    background=0, VOC labels=1..20.
    """
    objects = voc_target['annotation'].get('object', [])
    if isinstance(objects, dict):
        objects = [objects]

    boxes, labels = [], []
    for obj in objects:
        label = VOC_CLASS_TO_LABEL.get(obj['name'], 0)
        if label == 0:
            continue
        b = obj['bndbox']
        x1, y1 = float(b['xmin']), float(b['ymin'])
        x2, y2 = float(b['xmax']), float(b['ymax'])
        if x2 <= x1 or y2 <= y1:
            continue
        boxes.append([x1, y1, x2, y2])
        labels.append(label)

    if not boxes:
        return {
            'boxes':  torch.zeros((0, 4), dtype=torch.float32, device=device),
            'labels': torch.zeros((0,),   dtype=torch.int64,   device=device)
        }
    return {
        'boxes':  torch.tensor(boxes,  dtype=torch.float32, device=device),
        'labels': torch.tensor(labels, dtype=torch.int64,   device=device)
    }


def _box_iou(b1, b2):
    ix1, iy1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    ix2, iy2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0.0


def compute_voc_metrics(model, val_loader, num_classes, class_counts,
                        iou_thresh=0.5, max_batches=250):
    """
    VOC-style AP50 (IoU=0.5, 11-point interpolation) per class.
    class_counts: 20-element list (0-indexed).
    Returns mAP, per-class AP, Head/Mid/Tail AP.
    """
    model.eval()
    class_dets = defaultdict(list)  # cidx(0-19) -> [(score, is_tp), ...]
    class_n_gt = defaultdict(int)   # cidx(0-19) -> # GT boxes

    with torch.no_grad():
        for batch_idx, batch_data in enumerate(val_loader):
            if batch_idx >= max_batches:
                break
            images, gt_list = [], []
            for img, target in batch_data:
                images.append(img.to(DEVICE))
                gt_list.append(convert_voc_target(target, device='cpu'))

            try:
                preds = model(images)
            except Exception as e:
                print(f'Warning batch {batch_idx}: {str(e)[:80]}')
                continue

            for pred, gt in zip(preds, gt_list):
                gt_boxes  = gt['boxes'].numpy()
                gt_labels = gt['labels'].numpy()  # values 1-20

                for lbl in gt_labels:
                    class_n_gt[int(lbl) - 1] += 1  # 0-indexed

                pred_boxes  = pred['boxes'].cpu().numpy()
                pred_labels = pred['labels'].cpu().numpy()
                pred_scores = pred['scores'].cpu().numpy()

                gt_matched = set()
                for i in np.argsort(-pred_scores):
                    lbl = int(pred_labels[i])
                    if lbl == 0 or pred_scores[i] < 0.01:
                        continue
                    cidx = lbl - 1
                    best_iou, best_j = iou_thresh, -1
                    for j in range(len(gt_boxes)):
                        if int(gt_labels[j]) != lbl or j in gt_matched:
                            continue
                        iou = _box_iou(pred_boxes[i], gt_boxes[j])
                        if iou > best_iou:
                            best_iou, best_j = iou, j
                    is_tp = best_j >= 0
                    if is_tp:
                        gt_matched.add(best_j)
                    class_dets[cidx].append((float(pred_scores[i]), is_tp))

    # 11-point AP per class
    per_class_ap = []
    for cidx in range(20):
        n_gt = class_n_gt.get(cidx, 0)
        dets = class_dets.get(cidx, [])
        if n_gt == 0 or not dets:
            per_class_ap.append(0.0)
            continue
        dets.sort(key=lambda x: -x[0])
        tp_cum = np.cumsum([1 if d[1] else 0 for d in dets])
        fp_cum = np.cumsum([0 if d[1] else 1 for d in dets])
        recalls    = tp_cum / n_gt
        precisions = tp_cum / (tp_cum + fp_cum)
        ap = sum(
            precisions[recalls >= t].max() if np.any(recalls >= t) else 0.0
            for t in np.linspace(0, 1, 11)
        ) / 11
        per_class_ap.append(float(ap))

    mAP    = float(np.mean(per_class_ap))
    counts = np.array(class_counts)
    q1, q3 = np.percentile(counts, [33, 66])

    head_idxs = [i for i in range(20) if counts[i] >= q3]
    mid_idxs  = [i for i in range(20) if q1 <= counts[i] < q3]
    tail_idxs = [i for i in range(20) if counts[i] < q1]

    head_ap = float(np.mean([per_class_ap[i] for i in head_idxs])) if head_idxs else 0.0
    mid_ap  = float(np.mean([per_class_ap[i] for i in mid_idxs]))  if mid_idxs  else 0.0
    tail_ap = float(np.mean([per_class_ap[i] for i in tail_idxs])) if tail_idxs else 0.0

    return {
        'mAP':          mAP,
        'AP50':         mAP,
        'Per_Class_AP': per_class_ap,
        'Head_AP':      head_ap,
        'Mid_AP':       mid_ap,
        'Tail_AP':      tail_ap,
    }

print('\u2713 Model and evaluation functions defined')

In [ ]:
# Cell 4: Training Function (not executed here)

def train_model(loss_name, class_counts, train_loader, val_loader, num_classes,
               alpha=1.0, gamma=2.0, epochs=20):
    import torchvision.models.detection.roi_heads as _roi_module
    import torch.nn.functional as _F

    # ── Monkey-patch fastrcnn_loss to inject custom classification loss ──
    _orig_loss = _roi_module.fastrcnn_loss

    if get_clf_loss is not None and loss_name != 'ce':
        # RoI head sees num_classes outputs (0=bg, 1..20=VOC objects).
        # Prepend large background count so bg gets low weight.
        roi_counts = [int(sum(class_counts) * 10)] + list(class_counts)  # len=21
        _clf_fn = get_clf_loss(loss_name, roi_counts, alpha=alpha, gamma=gamma)

        def _patched(class_logits, box_regression, labels, regression_targets):
            labels_cat = torch.cat(labels, dim=0)
            reg_cat    = torch.cat(regression_targets, dim=0)
            cls_loss   = _clf_fn(class_logits, labels_cat)
            pos_idx    = torch.where(labels_cat > 0)[0]
            if len(pos_idx) == 0:
                box_loss = torch.tensor(0.0, device=class_logits.device,
                                        requires_grad=True)
            else:
                pos_labels = labels_cat[pos_idx]
                N, _ = class_logits.shape
                br = box_regression.reshape(N, box_regression.size(-1) // 4, 4)
                box_loss = _F.smooth_l1_loss(
                    br[pos_idx, pos_labels], reg_cat[pos_idx],
                    beta=1/9, reduction='sum'
                )
                box_loss = box_loss / labels_cat.numel()
            return cls_loss, box_loss

        _roi_module.fastrcnn_loss = _patched

    try:
        model = build_detector(num_classes)
        model = model.to(DEVICE)

        # Differential LR: lower for pretrained backbone, higher for new head
        backbone_params = [p for n, p in model.named_parameters()
                           if 'box_predictor' not in n]
        head_params     = [p for n, p in model.named_parameters()
                           if 'box_predictor' in n]
        optimizer = optim.SGD(
            [{'params': backbone_params, 'lr': 0.0001},
             {'params': head_params,     'lr': 0.001}],
            momentum=0.9, weight_decay=1e-4
        )
        # LR drop at epoch 12 and 17
        scheduler = optim.lr_scheduler.MultiStepLR(
            optimizer, milestones=[12, 17], gamma=0.1
        )

        history  = {'train_loss': [], 'val_mAP': []}
        best_mAP = 0.0

        print(f"\n{'='*50}\nTraining {loss_name.upper()} "
              f"(\u03b1={alpha:.2f}, \u03b3={gamma:.2f})\n{'='*50}")

        for epoch in range(epochs):
            model.train()
            epoch_loss, batch_count, skipped = 0.0, 0, 0

            for batch_idx, batch_data in enumerate(train_loader):
                if MAX_TRAIN_BATCHES and batch_idx >= MAX_TRAIN_BATCHES:
                    break

                images, targets_list, valid = [], [], True
                try:
                    for img, target in batch_data:
                        if img is None:
                            valid = False; break
                        if torch.isnan(img).any() or torch.isinf(img).any():
                            valid = False; break
                        images.append(img.to(DEVICE))
                        targets_list.append(
                            convert_voc_target(target, device=DEVICE)
                        )

                    if not valid or not images:
                        skipped += 1; continue

                    loss_dict = model(images, targets_list)
                    losses    = sum(v for v in loss_dict.values())

                    if torch.isnan(losses) or torch.isinf(losses):
                        skipped += 1; continue

                    optimizer.zero_grad()
                    losses.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

                    epoch_loss  += losses.item()
                    batch_count += 1

                except Exception as e:
                    skipped += 1
                    if batch_idx < 3:
                        print(f'  Batch {batch_idx} error: {str(e)[:80]}')
                    continue

            avg_loss = epoch_loss / max(batch_count, 1)
            history['train_loss'].append(avg_loss)

            # Validate every 5 epochs and at the last epoch
            if (epoch + 1) % 5 == 0 or epoch == epochs - 1:
                metrics = compute_voc_metrics(
                    model, val_loader, num_classes, class_counts,
                    max_batches=100  # 400 val images for speed
                )
                mAP = metrics['mAP']
                history['val_mAP'].append(mAP)
                if mAP > best_mAP:
                    best_mAP = mAP
                print(f'Epoch {epoch+1:2d}/{epochs} | Loss: {avg_loss:.4f} | '
                      f'AP50: {mAP:.4f} | Skipped: {skipped}')

            scheduler.step()

        print(f'\u2713 {loss_name.upper()} best AP50: {best_mAP:.4f}')
        return model, history, best_mAP

    finally:
        _roi_module.fastrcnn_loss = _orig_loss  # always restore

print('\u2713 train_model defined (not executed yet)')

In [ ]:
# Cell 5: Optuna Hyperparameter Search
import optuna
from optuna.samplers import GridSampler
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ['TQDM_DISABLE'] = '1'

print(f'Optuna search (proxy: {PROXY_EPOCHS} epochs, {MAX_TRAIN_BATCHES} batches/epoch)\n')
optuna_best = {}

# ── PWCE: alpha in [0.5, 5.0] ─────────────────────────────────────
N_PWCE = 10
study_pwce = optuna.create_study(
    direction='maximize',
    sampler=GridSampler({'alpha': np.linspace(0.5, 5.0, N_PWCE).tolist()})
)
def obj_pwce(trial):
    alpha = trial.suggest_float('alpha', 0.5, 5.0)
    _, _, mAP = train_model('pwce', class_counts, train_loader, val_loader,
                            NUM_CLASSES, alpha=alpha, epochs=PROXY_EPOCHS)
    return mAP if mAP is not None else 0.0
study_pwce.optimize(obj_pwce, n_trials=N_PWCE)
best_pwce = study_pwce.best_params
optuna_best['pwce'] = best_pwce
print(f'\u2713 PWCE  best: alpha={best_pwce["alpha"]:.2f}  (AP50={study_pwce.best_value:.4f})')

# ── PLWCE: alpha in [2.0, 15.0] ───────────────────────────────────
N_PLWCE = 10
study_plwce = optuna.create_study(
    direction='maximize',
    sampler=GridSampler({'alpha': np.linspace(2.0, 15.0, N_PLWCE).tolist()})
)
def obj_plwce(trial):
    alpha = trial.suggest_float('alpha', 2.0, 15.0)
    _, _, mAP = train_model('plwce', class_counts, train_loader, val_loader,
                            NUM_CLASSES, alpha=alpha, epochs=PROXY_EPOCHS)
    return mAP if mAP is not None else 0.0
study_plwce.optimize(obj_plwce, n_trials=N_PLWCE)
best_plwce = study_plwce.best_params
optuna_best['plwce'] = best_plwce
print(f'\u2713 PLWCE best: alpha={best_plwce["alpha"]:.2f}  (AP50={study_plwce.best_value:.4f})')

# ── FOCAL: gamma in [0.5, 5.0] ────────────────────────────────────
N_FOCAL = 10
study_focal = optuna.create_study(
    direction='maximize',
    sampler=GridSampler({'gamma': np.linspace(0.5, 5.0, N_FOCAL).tolist()})
)
def obj_focal(trial):
    gamma = trial.suggest_float('gamma', 0.5, 5.0)
    _, _, mAP = train_model('focal', class_counts, train_loader, val_loader,
                            NUM_CLASSES, gamma=gamma, epochs=PROXY_EPOCHS)
    return mAP if mAP is not None else 0.0
study_focal.optimize(obj_focal, n_trials=N_FOCAL)
best_focal = study_focal.best_params
optuna_best['focal'] = best_focal
print(f'\u2713 FOCAL best: gamma={best_focal["gamma"]:.2f}  (AP50={study_focal.best_value:.4f})')

print(f"\nOptuna Results:\n{json.dumps(optuna_best, indent=2)}")

In [ ]:
# Cell 6: Full Experiment
LOSS_CONFIGS = ['ce', 'pwce', 'lwce', 'plwce', 'cb', 'focal']
all_results   = {}
all_histories = {}

print(f"\n{'='*60}")
print(f'FULL EXPERIMENT: {FINAL_EPOCHS} epochs x {len(LOSS_CONFIGS)} loss functions')
print(f"{'='*60}\n")

for loss_name in LOSS_CONFIGS:
    alpha = optuna_best.get(loss_name, {}).get('alpha', 1.0)
    gamma = optuna_best.get('focal', {}).get('gamma', 2.0) if loss_name == 'focal' else 2.0

    model, history, _ = train_model(
        loss_name, class_counts, train_loader, val_loader,
        NUM_CLASSES, alpha=alpha, gamma=gamma, epochs=FINAL_EPOCHS
    )

    # Final evaluation on full VOC 2007 test set (~5K images = 1250 batches)
    metrics = compute_voc_metrics(model, val_loader, NUM_CLASSES, class_counts,
                                  max_batches=1250)
    all_results[loss_name]   = metrics
    all_histories[loss_name] = history

    print(f'\n{loss_name.upper()} Final:')
    print(f'  AP50:    {metrics["mAP"]:.4f}')
    print(f'  Head AP: {metrics["Head_AP"]:.4f}')
    print(f'  Mid AP:  {metrics["Mid_AP"]:.4f}')
    print(f'  Tail AP: {metrics["Tail_AP"]:.4f}')
    per_cls = {VOC_CLASSES[i]: f'{v:.3f}'
               for i, v in enumerate(metrics['Per_Class_AP'])}
    print(f'  Per-class: {per_cls}')

print(f"\n{'='*60}")
print('\u2713 Full experiment completed!')
print(f"{'='*60}")

In [ ]:
# Cell 7: Visualization + Save Results
loss_names = list(all_results.keys())
mAPs     = [all_results[ln]['mAP']     for ln in loss_names]
head_aps = [all_results[ln]['Head_AP'] for ln in loss_names]
mid_aps  = [all_results[ln]['Mid_AP']  for ln in loss_names]
tail_aps = [all_results[ln]['Tail_AP'] for ln in loss_names]
colors   = plt.cm.Set2(np.linspace(0, 1, len(loss_names)))

# ── Bar chart (mAP + Head/Mid/Tail) ──────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('VOC 2007 Detection Results \u2014 Loss Function Comparison', fontsize=13)
for ax, vals, title in zip(
    axes,
    [mAPs, head_aps, mid_aps, tail_aps],
    ['mAP (AP50)', 'Head AP', 'Mid AP', 'Tail AP']
):
    bars = ax.bar(loss_names, vals, color=colors, alpha=0.85)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('AP')
    ax.grid(True, alpha=0.3, axis='y')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Per-class AP heatmap ──────────────────────────────────────────
per_cls_mat = np.array([all_results[ln]['Per_Class_AP'] for ln in loss_names])
fig, ax = plt.subplots(figsize=(15, 4))
im = ax.imshow(per_cls_mat, aspect='auto', cmap='RdYlGn', vmin=0, vmax=0.8)
ax.set_xticks(range(20))
ax.set_xticklabels(VOC_CLASSES, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(loss_names)))
ax.set_yticklabels([ln.upper() for ln in loss_names])
ax.set_title('Per-Class AP50 by Loss Function')
plt.colorbar(im, ax=ax, label='AP50')
for i in range(len(loss_names)):
    for j in range(20):
        ax.text(j, i, f'{per_cls_mat[i,j]:.2f}', ha='center', va='center',
                fontsize=6, color='black')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/per_class_ap_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Training curves ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ln in loss_names:
    h = all_histories[ln]
    axes[0].plot(h['train_loss'], label=ln.upper())
    ep_ticks = np.linspace(0, FINAL_EPOCHS, len(h['val_mAP']))
    axes[1].plot(ep_ticks, h['val_mAP'], label=ln.upper(), marker='o', markersize=4)
for ax, title, ylabel in zip(axes,
    ['Training Loss', 'Validation AP50'],
    ['Loss', 'AP50']):
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Save JSON ────────────────────────────────────────────────────
results_json = {
    'metadata': {
        'dataset':           'Pascal VOC 2007',
        'model':             'Faster-RCNN ResNet50+FPN (COCO pretrained)',
        'num_classes':       NUM_CLASSES,
        'final_epochs':      FINAL_EPOCHS,
        'max_train_batches': MAX_TRAIN_BATCHES,
        'optuna_best':       optuna_best,
    },
    'results': {
        ln: {k: v for k, v in all_results[ln].items()}
        for ln in loss_names
    }
}
with open(f'{RESULTS_DIR}/voc2007_results.json', 'w') as f:
    json.dump(results_json, f, indent=2)
print(f'\u2713 Saved JSON')

# ── Save Excel ───────────────────────────────────────────────────
summary_df = pd.DataFrame({
    'Loss':    loss_names,
    'mAP_AP50': mAPs,
    'Head_AP': head_aps,
    'Mid_AP':  mid_aps,
    'Tail_AP': tail_aps,
})
per_class_df = pd.DataFrame(
    {ln: all_results[ln]['Per_Class_AP'] for ln in loss_names},
    index=VOC_CLASSES
)
history_rows = [
    {'loss': ln, 'epoch': ep + 1, 'train_loss': v}
    for ln in loss_names
    for ep, v in enumerate(all_histories[ln]['train_loss'])
]
history_df = pd.DataFrame(history_rows)

with pd.ExcelWriter(f'{RESULTS_DIR}/voc2007_results.xlsx', engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='Summary',          index=False)
    per_class_df.to_excel(writer, sheet_name='Per_Class_AP')
    history_df.to_excel(writer, sheet_name='Training_History', index=False)
print(f'\u2713 Saved Excel')

# ── Summary print ────────────────────────────────────────────────
print(f"\n{'='*60}")
print('EXPERIMENT SUMMARY')
print(f"{'='*60}")
print(summary_df.to_string(index=False))
print(f"{'='*60}")

# ── GitHub push ──────────────────────────────────────────────────
print('\nPushing to GitHub...')
try:
    os.chdir(REPO_PATH)
    os.system('git config user.email "colab@example.com"')
    os.system('git config user.name "Colab"')
    os.system('git add image_detection/results/VOC2007/ 2>/dev/null || true')
    os.system('git commit -m "Add VOC2007 detection results" 2>/dev/null || echo "No changes"')
    os.system('git push 2>/dev/null || echo "Push failed (auth required)"')
    print('\u2713 GitHub push completed!')
except Exception as e:
    print(f'GitHub push skipped: {e}')

print(f'\n\u2713 All results saved to: {RESULTS_DIR}')